# Evaluation — exact match per model and prompting strategy

Precision / Recall / F1 for exact match, computed key by key over the annotation JSON
and micro-averaged over **all** the notes of each (model, strategy).

* one **slot** = one key: `flag_is_medication_completed` + the 12 attributes of each medication;
* **TP** = gold has a value and the prediction reproduces it exactly;
  **FP** = the prediction has a value that is not the gold one;
  **FN** = gold has a value that the prediction does not reproduce;
* `null` on both sides is a true negative and is not counted;
* medications are compared position by position.

**Failed extractions** (files under `<model>/errors/<strategy>/`) are loaded like any
other and marked `status="error"`. In the headline table they count as *no output*:
every annotated slot of that note becomes a false negative. This penalises recall and
never rewards precision, so a model that crashes on the hard notes cannot look better
than one that answers them badly. The `invalid_rate` column and the valid-only table in
section 5 report the other half of the picture.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

from clinical_notes_extraction.utils.llm.evaluation import (  # noqa: E402
    ATTRIBUTES,
    TOP_LEVEL_KEYS,
    evaluate_all,
    load_ground_truth_dir,
    load_results_tree,
    normalise,
)

from clinical_notes_extraction.utils.llm.evaluation import (  # noqa: E402
    metrics_table, metrics_to_docx, metrics_to_html, metrics_to_latex,
)



In [ ]:
NOTEBOOK_DIR = Path.cwd()
ROOT = next(p for p in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT / "src"))

pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
print("project root:", ROOT)


## 1. Paths

`RUN_ID` pins one extraction run; never mix timestamps in the same table.

In [ ]:
DATA_DIR = NOTEBOOK_DIR / "data"

SPLIT = "dev"          # ground_truth/dev | ground_truth/prod
CRITERION = "exact_match"        # "exact_match" no notebook 5

GROUND_TRUTH_DIR = DATA_DIR / "annotations" / "ground_truth" / SPLIT
RESULTS_ROOT = DATA_DIR / "llm_extraction_results" / SPLIT
RUN_ID = "20260813_165826"
RUN_DIR = RESULTS_ROOT / RUN_ID
# RUN_ID = sorted(p.name for p in RESULTS_ROOT.iterdir() if p.is_dir())[-1]  # latest run

OUTPUT_DIR = RESULTS_ROOT / "old_evaluation" / RUN_ID / CRITERION
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert GROUND_TRUTH_DIR.is_dir(), GROUND_TRUTH_DIR
assert RUN_DIR.is_dir(), RUN_DIR

## 2. Load

`load_ground_truth_dir` reads one annotation per file (the note id comes from the
`note_id` key, falling back to the file name). `load_results_tree` walks
`<run>/<model>/<strategy>/*.json`, skips `prompts/`, and picks up
`<model>/errors/<strategy>/*.json` as failed records.

In [ ]:
gold_by_id = load_ground_truth_dir(GROUND_TRUTH_DIR)
records = load_results_tree(RUN_DIR)

print(f"{len(gold_by_id)} annotated notes in {SPLIT}")
print(f"{len(records)} result files in run {RUN_ID}")

## 3. Coverage check

Before reading any metric: does every (model, strategy) cover every annotated note?
`missing` are notes with neither an output file nor an error file — they are also
scored as no output, but they usually mean the grid did not finish.

In [ ]:
records_df = pd.DataFrame(records)
annotated = set(gold_by_id)

coverage = (
    records_df[records_df.note_id.isin(annotated)]
    .groupby(["model", "strategy"])
    .agg(ok=("status", lambda s: (s == "ok").sum()),
         errors=("status", lambda s: (s != "ok").sum()))
)
coverage["missing"] = len(annotated) - coverage.ok - coverage.errors
coverage["not_annotated"] = (
    records_df[~records_df.note_id.isin(annotated)]
    .groupby(["model", "strategy"]).size()
)
coverage.fillna(0).astype(int)

## 4. Headline table — errors counted as no output

In [ ]:
results = evaluate_all(gold_by_id, records)

runs = (
    pd.DataFrame([run.record() for run in results.values()])
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)
runs[["model", "strategy", "precision", "recall", "f1",
      "tp", "fp", "fn", "support", "n_notes", "invalid_rate"]]

In [ ]:
for metric in ["precision", "recall", "f1"]:
    print(f"\n=== {metric} ===")
    print(runs.pivot(index="model", columns="strategy", values=metric).round(3))

## 5. The two secondary views

`valid_only` drops the failed notes entirely: quality *conditional* on producing
parseable JSON. Read it together with `invalid_rate` — a high `f1_valid_only` with a
high `invalid_rate` means a model that is accurate when it answers and often does not.

`casefold` ignores capitalisation, which shows how much of the strict error is
`albuterol sulfate` vs `Albuterol Sulfate` rather than a real extraction mistake.

In [ ]:
valid_only = evaluate_all(gold_by_id, records, skip_invalid=True)
casefolded = evaluate_all(gold_by_id, records, casefold=True)

def f1_series(res, name):
    return (
        pd.DataFrame([r.record() for r in res.values()])
        .set_index(["model", "strategy"])["f1"].rename(name)
    )

comparison = (
    runs.set_index(["model", "strategy"])[["f1", "invalid_rate"]]
    .rename(columns={"f1": "f1_strict"})
    .join(f1_series(valid_only, "f1_valid_only"))
    .join(f1_series(casefolded, "f1_casefold"))
)
comparison["gap_errors"] = comparison.f1_valid_only - comparison.f1_strict
comparison["gap_casing"] = comparison.f1_casefold - comparison.f1_strict
comparison.sort_values("f1_strict", ascending=False).round(3)

## 6. Breakdown per key

In [ ]:
per_key = pd.DataFrame([row for run in results.values() for row in run.per_key_records()])

per_key.pivot_table(index="key", columns=["model", "strategy"], values="f1").reindex(
    list(TOP_LEVEL_KEYS) + list(ATTRIBUTES)
).round(3)

In [ ]:
# keys that are almost always null have very little support and unstable F1
per_key.groupby("key").agg(support=("support", "max"), mean_f1=("f1", "mean")).reindex(
    list(TOP_LEVEL_KEYS) + list(ATTRIBUTES)
).round(3)

## 7. Breakdown per note

In [ ]:
per_note = pd.DataFrame([row for run in results.values() for row in run.per_note_records()])

per_note.groupby("note_id").agg(
    mean_f1=("f1", "mean"),
    failed_runs=("valid", lambda s: (~s).sum()),
    n_gold_medications=("n_gold_medications", "max"),
).sort_values("mean_f1").head(10).round(3)

### Which slots disagree, for one run and one note

In [ ]:
MODEL, STRATEGY = runs.loc[0, "model"], runs.loc[0, "strategy"]
NOTE_ID = per_note[(per_note.model == MODEL) & (per_note.strategy == STRATEGY)] \
    .sort_values("f1").iloc[0].note_id

prediction = next(
    (r.get("output") or {} for r in records
     if r["model"] == MODEL and r["strategy"] == STRATEGY and r["note_id"] == NOTE_ID),
    {},
)

rows = [
    {"medication": "-", "key": key,
     "gold": normalise(gold_by_id[NOTE_ID].get(key)),
     "predicted": normalise(prediction.get(key))}
    for key in TOP_LEVEL_KEYS
]

gold_meds = gold_by_id[NOTE_ID].get("medications") or []
pred_meds = prediction.get("medications") or []
for i in range(max(len(gold_meds), len(pred_meds))):
    g = (gold_meds[i] if i < len(gold_meds) else {}).get("attributes", {})
    p = (pred_meds[i] if i < len(pred_meds) else {}).get("attributes", {})
    for key in ATTRIBUTES:
        rows.append({"medication": i + 1, "key": key,
                     "gold": normalise(g.get(key)), "predicted": normalise(p.get(key))})

diff = pd.DataFrame(rows)
print(f"{MODEL} / {STRATEGY} — note {NOTE_ID}")
diff[diff.gold != diff.predicted]

## 8. Final table — one row per key, P/R/F1 per model and prompting strategy

Same shape as the reference table: keys as rows, `P | R | F1` inside each column block.

* keys that are never annotated (no gold value anywhere) are dropped — an all-zero row
  says nothing about the model, only that the attribute does not occur in this split;
* **Macro** = unweighted mean over the keys shown, so a rare key weighs as much as
  `active_substance`;
* **Micro** = counts of those keys pooled, which is the headline number of section 4;
* failed extractions are still counted as missing output.

`models=` / `strategies=` restrict the columns, which gives one table per model or one
per prompting strategy; the column level that becomes constant is dropped automatically.

In [ ]:
table_all = metrics_table(results)
table_all.map("{:.2f}".format)          # two decimals, like the reference table

In [ ]:
MODELS = sorted({model for model, _ in results})
STRATEGIES = sorted({strategy for _, strategy in results})

for model in MODELS:
    print(f"\n=== {model} ===")
    print(metrics_table(results, models=[model]).map("{:.2f}".format).to_string())

In [ ]:
for strategy in STRATEGIES:
    print(f"\n=== {strategy} ===")
    print(metrics_table(results, strategies=[strategy]).map("{:.2f}".format).to_string())

In [ ]:
TABLES_DIR = OUTPUT_DIR / "tables"
TABLES_DIR.mkdir(exist_ok=True)

def slug(name: str) -> str:
    return name.replace(":", "_").replace(".", "_")

# headline table, indexed so that the index becomes the leading columns in Word
headline = runs.set_index(["model", "strategy"])[
    ["precision", "recall", "f1", "invalid_rate"]
]
headline.columns = ["P", "R", "F1", "invalid"]

# (filename, caption, dataframe)
exports = [
    ("summary", "Table 1: Exact-match Precision (P), Recall (R) and F1 per model and "
                "prompting strategy, over all annotated notes. Failed extractions "
                "counted as missing output.", headline),
]
exports += [
    (f"by_key_{slug(strategy)}",
     f"Table: exact-match results per key under {strategy} prompting.",
     metrics_table(results, strategies=[strategy]))
    for strategy in STRATEGIES
]
exports += [
    (f"by_key_{slug(model)}",
     f"Table: exact-match results per key for {model}.",
     metrics_table(results, models=[model]))
    for model in MODELS
]
exports.append(("appendix_full_grid",
                "Appendix table: every model and prompting strategy.", table_all))

tables = {caption: frame for _, caption, frame in exports}

DOCX_PATH = f"{TABLES_DIR}/exact_match_tables_{RUN_ID}.docx"
try:
    metrics_to_docx(tables, DOCX_PATH, title=f"Exact-match evaluation — run {RUN_ID}")
    print("wrote", DOCX_PATH)
except ImportError:
    print("python-docx not installed (pip install python-docx) — writing HTML instead")

# HTML copies: open in a browser and paste into Word; works without python-docx
for name, caption, frame in exports:
    (TABLES_DIR / f"{name}.html").write_text(
        metrics_to_html(frame, caption=caption), encoding="utf-8"
    )

# LaTeX, if you ever need it:
# (TABLES_DIR / "exact_match_all.tex").write_text(metrics_to_latex(table_all), encoding="utf-8")

print(sorted(p.name for p in TABLES_DIR.iterdir()))

## 9. Export the raw numbers

In [ ]:
runs.to_csv(OUTPUT_DIR / "metrics_per_run.csv", index=False)
per_key.to_csv(OUTPUT_DIR / "metrics_per_key.csv", index=False)
per_note.to_csv(OUTPUT_DIR / "metrics_per_note.csv", index=False)
coverage.to_csv(OUTPUT_DIR / "coverage.csv")

print("written to", OUTPUT_DIR)
print(sorted(p.name for p in OUTPUT_DIR.glob("*.csv")))